# Notebook 1: BDD100K Dataset Preparation
**Project:** Vehicle Detection — YOLO vs. DETR  
**Authors:** David Ho, Mahmoud Abdulkareem

**Strategy:**
- BDD100K zip lives on **Drive** (download once, never again)
- Images unzipped to **local Colab disk** (fast reads during training)
- Generated label files backed up to **Drive** (restored in < 1 min each session)
- Models and results always saved to **Drive**

> Run this notebook first, once. After that, Notebooks 2–4 handle their own session setup.

## Step 1: Mount Drive + Check GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — go to Runtime > Change runtime type > T4 GPU')

## Step 2: Install Dependencies

In [ ]:
!pip install -q pillow matplotlib tqdm

In [ ]:
import json, shutil, random, zipfile, os
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm import tqdm
random.seed(42)

## Step 3: Config

In [ ]:
# Drive paths (persistent)
DRIVE_ROOT    = Path('/content/drive/MyDrive/vehicle_detection')
DRIVE_ZIP     = DRIVE_ROOT / 'downloads' / 'solesensei_bdd100k.zip'
DRIVE_BACKUP  = DRIVE_ROOT / 'processed'   # label txt files + COCO JSONs
DRIVE_RESULTS = DRIVE_ROOT / 'results'

# Local paths (fast, session-only)
LOCAL_BDD  = Path('/content/bdd100k')   # raw images live here
LOCAL_DATA = Path('/content/data')      # YOLO labels + COCO JSONs

VEHICLE_CLASSES = ['car', 'truck', 'bus', 'motorcycle']
CLASS_TO_ID     = {cls: i for i, cls in enumerate(VEHICLE_CLASSES)}
IMG_W, IMG_H    = 1280, 720

for d in [DRIVE_ROOT, DRIVE_BACKUP, DRIVE_RESULTS]:
    d.mkdir(parents=True, exist_ok=True)
print('Class map:', CLASS_TO_ID)

## Step 4: Download BDD100K from Kaggle (runs once)

In [ ]:
if DRIVE_ZIP.exists():
    print('Zip already on Drive — skipping download.')
else:
    DRIVE_ZIP.parent.mkdir(parents=True, exist_ok=True)
    from google.colab import files
    print('Upload your kaggle.json when prompted...')
    files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    print('Downloading BDD100K (~7.6 GB)...')
    !kaggle datasets download -d solesensei/solesensei_bdd100k -p "{DRIVE_ZIP.parent}"
    print('Download complete.')

## Step 5: Unzip to Local Disk (~5 min, fast)

In [ ]:
if LOCAL_BDD.exists():
    print('Already unzipped locally.')
else:
    print('Unzipping to local disk (~5 min)...')
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
        z.extractall('/content/')
    print('Unzip done.')

# Auto-detect extra nesting (Kaggle sometimes adds an extra bdd100k/ folder)
if not (LOCAL_BDD / 'images').exists() and (LOCAL_BDD / 'bdd100k' / 'images').exists():
    LOCAL_BDD = LOCAL_BDD / 'bdd100k'
    print('Detected extra nesting — adjusted LOCAL_BDD to:', LOCAL_BDD)

IMG_DIR   = LOCAL_BDD / 'images' / '100k'
LABEL_DIR = LOCAL_BDD / 'labels' / 'det_20'

assert IMG_DIR.exists(),   f'Images not found: {IMG_DIR}'
assert LABEL_DIR.exists(), f'Labels not found: {LABEL_DIR}'
print('Paths OK')
print('  Images:', IMG_DIR)
print('  Labels:', LABEL_DIR)

## Step 6: Load and Explore Annotations

In [ ]:
with open(LABEL_DIR / 'det_train.json') as f:
    train_anns = json.load(f)
with open(LABEL_DIR / 'det_val.json') as f:
    val_anns = json.load(f)

print(f'Train frames : {len(train_anns):,}')
print(f'Val   frames : {len(val_anns):,}')

all_cats = Counter()
for frame in train_anns:
    for lbl in (frame.get('labels') or []):
        all_cats[lbl['category']] += 1

print('\nCategory distribution (train):')
for cat, cnt in all_cats.most_common():
    marker = ' <-- KEEP' if cat in VEHICLE_CLASSES else ''
    print(f'  {cat:20s} {cnt:>8,}{marker}')

## Step 7: Filter to Vehicle Classes

In [ ]:
def filter_frames(frames, keep_classes):
    kept = []
    for frame in frames:
        labels = [l for l in (frame.get('labels') or []) if l['category'] in keep_classes]
        if labels:
            kept.append({**frame, 'labels': labels})
    return kept

train_filtered = filter_frames(train_anns, VEHICLE_CLASSES)
val_filtered   = filter_frames(val_anns,   VEHICLE_CLASSES)

random.shuffle(val_filtered)
split_idx  = int(len(val_filtered) * 0.8)
val_split  = val_filtered[:split_idx]
test_split = val_filtered[split_idx:]

print(f'Train : {len(train_filtered):,}')
print(f'Val   : {len(val_split):,}')
print(f'Test  : {len(test_split):,}')

## Step 8: Generate YOLO Labels
Images stay in their original BDD100K location. Only small `.txt` label files are created.

In [ ]:
YOLO_LBL = LOCAL_DATA / 'yolo' / 'labels'

def write_yolo_labels(frames, split, src_img_dir, lbl_dir, class_to_id):
    out = lbl_dir / split
    out.mkdir(parents=True, exist_ok=True)
    ok = 0
    for frame in tqdm(frames, desc=f'YOLO labels {split}'):
        if not (src_img_dir / frame['name']).exists():
            continue
        lines = []
        for lbl in frame['labels']:
            b  = lbl['box2d']
            cx = ((b['x1'] + b['x2']) / 2) / IMG_W
            cy = ((b['y1'] + b['y2']) / 2) / IMG_H
            w  = (b['x2'] - b['x1']) / IMG_W
            h  = (b['y2'] - b['y1']) / IMG_H
            lines.append(f"{class_to_id[lbl['category']]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
        (out / frame['name'].replace('.jpg', '.txt')).write_text('\n'.join(lines))
        ok += 1
    print(f'  {split}: {ok:,} label files')

# BDD100K val images are used for both val and test splits
write_yolo_labels(train_filtered, 'train', IMG_DIR / 'train', YOLO_LBL, CLASS_TO_ID)
write_yolo_labels(val_split,      'val',   IMG_DIR / 'val',   YOLO_LBL, CLASS_TO_ID)
write_yolo_labels(test_split,     'test',  IMG_DIR / 'val',   YOLO_LBL, CLASS_TO_ID)

In [ ]:
# Save which filenames belong to val vs test (needed to separate them at eval time)
split_names = {
    'val':  [f['name'] for f in val_split],
    'test': [f['name'] for f in test_split],
}
(LOCAL_DATA / 'yolo').mkdir(parents=True, exist_ok=True)
(LOCAL_DATA / 'yolo' / 'split_names.json').write_text(json.dumps(split_names))

# dataset.yaml — images point directly to BDD100K local folder
yaml_txt = f"""path: {LOCAL_BDD}
train: images/100k/train
val:   images/100k/val
test:  images/100k/val

nc: {len(VEHICLE_CLASSES)}
names: {VEHICLE_CLASSES}
"""
(LOCAL_DATA / 'yolo' / 'dataset.yaml').write_text(yaml_txt)
print(yaml_txt)

## Step 9: Generate COCO Annotations (for DETR)

In [ ]:
COCO_ANN = LOCAL_DATA / 'coco' / 'annotations'
COCO_ANN.mkdir(parents=True, exist_ok=True)

# Store which BDD100K image subfolder each split uses
COCO_IMG_DIRS = {
    'train': IMG_DIR / 'train',
    'val':   IMG_DIR / 'val',
    'test':  IMG_DIR / 'val',
}

def frames_to_coco(frames, split, src_img_dir, ann_dir, class_to_id):
    categories = [{'id': v, 'name': k} for k, v in class_to_id.items()]
    images, annotations = [], []
    ann_id = 0
    for img_id, frame in enumerate(tqdm(frames, desc=f'COCO {split}')):
        if not (src_img_dir / frame['name']).exists():
            continue
        # Store path relative to LOCAL_BDD so it works across sessions
        rel_path = str((src_img_dir / frame['name']).relative_to(LOCAL_BDD))
        images.append({'id': img_id, 'file_name': rel_path, 'width': IMG_W, 'height': IMG_H})
        for lbl in frame['labels']:
            b = lbl['box2d']
            w, h = b['x2'] - b['x1'], b['y2'] - b['y1']
            annotations.append({
                'id': ann_id, 'image_id': img_id,
                'category_id': class_to_id[lbl['category']],
                'bbox': [b['x1'], b['y1'], w, h],
                'area': w * h, 'iscrowd': 0,
            })
            ann_id += 1
    out = ann_dir / f'instances_{split}.json'
    with open(out, 'w') as f:
        json.dump({'images': images, 'annotations': annotations, 'categories': categories}, f)
    print(f'  {split}: {len(images):,} images, {len(annotations):,} annotations')

coco_splits = {
    'train': (train_filtered, IMG_DIR / 'train'),
    'val':   (val_split,      IMG_DIR / 'val'),
    'test':  (test_split,     IMG_DIR / 'val'),
}
for split, (frames, src_dir) in coco_splits.items():
    frames_to_coco(frames, split, src_dir, COCO_ANN, CLASS_TO_ID)

## Step 10: Backup Labels to Drive

In [ ]:
# Only labels + JSONs go to Drive (small). Images stay local.
if DRIVE_BACKUP.exists():
    shutil.rmtree(DRIVE_BACKUP)
shutil.copytree(LOCAL_DATA, DRIVE_BACKUP)
print(f'Backed up to {DRIVE_BACKUP}')

# Save LOCAL_BDD path so other notebooks know where images are
(DRIVE_BACKUP / 'local_bdd_path.txt').write_text(str(LOCAL_BDD))
print('Done. Notebook 1 complete.')

## Step 11: Visualize Sample Annotations

In [ ]:
COLORS = {'car': '#2196F3', 'truck': '#FF9800', 'bus': '#4CAF50', 'motorcycle': '#E91E63'}

sample_frames = random.sample(train_filtered, min(6, len(train_filtered)))
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
fig.suptitle('BDD100K — Vehicle Annotations (filtered)', fontsize=14)

for ax, frame in zip(axes.flat, sample_frames):
    img_path = IMG_DIR / 'train' / frame['name']
    if not img_path.exists():
        ax.set_visible(False); continue
    ax.imshow(Image.open(img_path))
    for lbl in frame['labels']:
        b = lbl['box2d']
        ax.add_patch(patches.Rectangle(
            (b['x1'], b['y1']), b['x2']-b['x1'], b['y2']-b['y1'],
            linewidth=2, edgecolor=COLORS[lbl['category']], facecolor='none'
        ))
        ax.text(b['x1'], b['y1']-4, lbl['category'],
                color=COLORS[lbl['category']], fontsize=8, fontweight='bold')
    ax.axis('off')

handles = [patches.Patch(color=c, label=cls) for cls, c in COLORS.items()]
fig.legend(handles=handles, loc='lower center', ncol=4)
plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / 'sample_annotations.png'), dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
split_counts = {}
for split, (frames, _) in coco_splits.items():
    cnt = Counter()
    for frame in frames:
        for lbl in frame['labels']:
            cnt[lbl['category']] += 1
    split_counts[split] = cnt

x, width = range(len(VEHICLE_CLASSES)), 0.25
fig, ax = plt.subplots(figsize=(9, 5))
for i, (split, cnt) in enumerate(split_counts.items()):
    ax.bar([p + i*width for p in x], [cnt[c] for c in VEHICLE_CLASSES], width, label=split)
ax.set_xticks([p + width for p in x])
ax.set_xticklabels(VEHICLE_CLASSES)
ax.set_ylabel('Instance count')
ax.set_title('Vehicle class distribution per split')
ax.legend()
plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / 'class_distribution.png'), dpi=120)
plt.show()